# Dataset 3 – SBS News RSS Snapshot

This notebook documents Dataset 3 for the AgeTogether FIT5120 Iteration 1 Recent News prototype. It uses one static snapshot of the official SBS News Australia RSS feed. It does not scrape article webpages, reproduce full article content or implement automated refresh.

## 1. Data Acquisition and Provenance

**Provider:** SBS News (Special Broadcasting Service Corporation)  
**Feed type:** Official SBS News Australia RSS feed  
**Official RSS URL:** https://www.sbs.com.au/news/topic/australia/feed  
**SBS RSS feed directory:** https://www.sbs.com.au/news/article/feeds/nbv1rs3kw  
**Local acquisition date:** 3 September 2026  
**Raw snapshot:** `data/raw/sbs_news_feed_snapshot.xml`  
**Raw-file SHA-256:** `5a4cf94543aca33dcbb38e74e8837672710fbc6626cad30d327aa7cc3dc6218b`

Dataset 3 supports AgeTogether Social → Recent News. RSS (Rich Site Summary) is a structured, machine-readable feed format. Iteration 1 deliberately uses a static snapshot so that the analysis and prototype evidence are reproducible; it does not claim that the feed is real-time in the prototype.

Only RSS metadata is retained: title, publication date, original article URL, source, category, GUID and the feed-provided short description where present. Full article bodies are not scraped or reproduced. Users must follow the original SBS article URL to read the source article.

The SBS RSS feed is official, but it is not identified here as CC-licensed open data. SBS terms state that SBS material is copyright protected and rights not expressly granted are reserved. This Iteration 1 artefact therefore uses attribution and links back to original articles, avoids full-content reproduction and must not be treated as a licence to republish SBS articles or images.

## 2. Data Understanding

The raw XML snapshot is loaded from a repository-relative path. The RSS items are parsed directly; article webpages are not requested.

In [ ]:
from pathlib import Path
import hashlib
import json
import xml.etree.ElementTree as ET

import pandas as pd


RAW_FILENAME = "sbs_news_feed_snapshot.xml"
SOURCE_NAME = "SBS News – Australia RSS"
SOURCE_URL = "https://www.sbs.com.au/news/topic/australia/feed"


def display(value):
    print(value.to_string())


def find_repository_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw" / RAW_FILENAME).exists():
            return candidate
    raise FileNotFoundError("Repository root containing the Dataset 3 RSS snapshot was not found.")


repo_root = find_repository_root(Path.cwd().resolve())
raw_path = repo_root / "data" / "raw" / RAW_FILENAME

with raw_path.open("rb") as raw_file:
    raw_sha256 = hashlib.sha256(raw_file.read()).hexdigest()

rss_root = ET.parse(raw_path).getroot()
channel = rss_root.find("channel")
rss_items = channel.findall("item")

records = []
for item in rss_items:
    categories = [category.text for category in item.findall("category") if category.text]
    records.append(
        {
            "title": item.findtext("title"),
            "published_date_raw": item.findtext("pubDate"),
            "article_url": item.findtext("link"),
            "summary": item.findtext("description"),
            "category": "; ".join(categories) if categories else None,
            "guid": item.findtext("guid"),
            "source": SOURCE_NAME,
            "source_feed_url": SOURCE_URL,
        }
    )

raw_news_df = pd.DataFrame(records)

print(f"Repository root: {repo_root}")
print(f"Raw snapshot: {raw_path.relative_to(repo_root)}")
print(f"SHA-256: {raw_sha256}")
print(f"Feed title: {channel.findtext('title')}")

In [ ]:
print(f"Total RSS item records: {len(raw_news_df)}")
print("\nAvailable parsed fields:")
print(raw_news_df.columns.tolist())
print("\nFirst five parsed metadata records:")
display(raw_news_df.head())

availability_summary = pd.DataFrame(
    {
        "field": raw_news_df.columns,
        "non_missing_count": [raw_news_df[column].notna().sum() for column in raw_news_df.columns],
        "missing_count": [raw_news_df[column].isna().sum() for column in raw_news_df.columns],
    }
)
display(availability_summary)

print(f"Duplicate titles: {raw_news_df['title'].duplicated().sum()}")
print(f"Duplicate article URLs: {raw_news_df['article_url'].duplicated().sum()}")

## 3. Data Quality and Cleaning

The cleaning step keeps source-provided metadata only. Publication dates are parsed safely, exact duplicate records are removed, then duplicate article URLs are removed. Missing values are not invented. The original title is preserved, and full article bodies are not fetched.

In [ ]:
news_cleaned_df = raw_news_df.copy()
news_cleaned_df["published_date_parsed"] = pd.to_datetime(
    news_cleaned_df["published_date_raw"], errors="coerce", utc=True
)
news_cleaned_df["published_date"] = news_cleaned_df["published_date_parsed"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

exact_duplicates_removed = news_cleaned_df.duplicated().sum()
news_cleaned_df = news_cleaned_df.drop_duplicates().copy()

duplicate_urls_removed = news_cleaned_df["article_url"].duplicated().sum()
news_cleaned_df = news_cleaned_df.drop_duplicates(subset=["article_url"], keep="first").copy()

news_cleaned_df = news_cleaned_df.drop(columns=["published_date_raw", "published_date_parsed"])

cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "raw RSS records",
            "publication dates parsed successfully",
            "publication date parsing failures",
            "exact duplicate records removed",
            "duplicate article URLs removed",
            "cleaned records",
        ],
        "value": [
            len(raw_news_df),
            news_cleaned_df["published_date"].notna().sum(),
            news_cleaned_df["published_date"].isna().sum(),
            exact_duplicates_removed,
            duplicate_urls_removed,
            len(news_cleaned_df),
        ],
    }
)
display(cleaning_summary)

## 4. AgeTogether Relevance Filtering

The following rule is a transparent Iteration 1 prototype rule, not a news recommendation model and not a statement that an article is suitable for every older adult.

First, an item is Included when its title or RSS summary contains an AgeTogether-relevant keyword/topic: older people, older Australians, seniors, ageing/aging, community, social isolation, loneliness, scams, online safety, online, digital, health, wellbeing, transport, Victoria, Melbourne, vaccine or COVID.

The direct title/summary rule returns too few records for a useful static Recent News prototype. Therefore, a second explicitly documented broadening rule also Includes items carrying the official RSS category `Australia`, `Health`, `Life` or `COVID-19`. This broadens the prototype to Australian public-interest context while still excluding records outside both the keywords and these official feed categories.

`Included` means only that an item is retained for the Iteration 1 Recent News prototype. It does not mean the article is recommended, personalised, verified for older adults, or currently the most important news.

In [ ]:
keyword_topics = [
    "older people",
    "older australians",
    "seniors",
    "ageing",
    "aging",
    "community",
    "social isolation",
    "loneliness",
    "scams",
    "online safety",
    "online",
    "digital",
    "health",
    "wellbeing",
    "transport",
    "victoria",
    "melbourne",
    "vaccine",
    "covid",
]
broad_category_topics = {"Australia", "Health", "Life", "COVID-19"}


def classify_news_record(record):
    searchable_text = f"{record['title'] or ''} {record['summary'] or ''}".lower()
    keyword_matches = [term for term in keyword_topics if term in searchable_text]
    categories = set(record["category"].split("; ")) if pd.notna(record["category"]) else set()
    category_matches = sorted(categories & broad_category_topics)

    if keyword_matches or category_matches:
        reason_parts = []
        if keyword_matches:
            reason_parts.append("title/summary keyword match: " + ", ".join(keyword_matches))
        if category_matches:
            reason_parts.append("official RSS category match: " + ", ".join(category_matches))
        return pd.Series(["Included", "Included because " + "; ".join(reason_parts) + "."])

    return pd.Series(
        [
            "Excluded",
            "Excluded because no documented keyword/topic or broad official RSS category matched.",
        ]
    )


news_classified_df = news_cleaned_df.copy()
news_classified_df[["news_relevance", "news_relevance_reason"]] = news_classified_df.apply(
    classify_news_record, axis=1
)

relevance_summary = (
    news_classified_df["news_relevance"]
    .value_counts()
    .rename_axis("news_relevance")
    .reset_index(name="record_count")
)
relevance_summary["percentage"] = relevance_summary["record_count"] / len(news_classified_df) * 100
display(relevance_summary)

## 5. Descriptive Analysis

This describes the static RSS snapshot only. Record counts do not represent popularity, importance, quality or user preference.

In [ ]:
category_counts = (
    news_classified_df.assign(category_value=news_classified_df["category"].fillna("Not available from the source").str.split("; "))
    .explode("category_value")["category_value"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="record_count")
)

print(f"Raw feed records: {len(raw_news_df)}")
print(f"Cleaned records: {len(news_cleaned_df)}")
print(f"Included records: {(news_classified_df['news_relevance'] == 'Included').sum()}")
print(
    "Publication date range: "
    f"{news_classified_df['published_date'].min()} to {news_classified_df['published_date'].max()}"
)
print("\nCategory/topic counts:")
display(category_counts)

## 6. Product Data View and Export

`agetogether_news_records_classified.csv` retains all cleaned metadata records plus relevance fields for audit evidence. `agetogether_news_prototype.csv` retains Included records only.

The product exports contain RSS metadata only. They do not contain article bodies, article images, invented descriptions, recommendation scores or claims about article suitability. Users should follow `article_url` to the original SBS article.

In [ ]:
classified_export_path = repo_root / "data" / "processed" / "agetogether_news_records_classified.csv"
prototype_export_path = repo_root / "data" / "processed" / "agetogether_news_prototype.csv"

prototype_columns = [
    "title",
    "published_date",
    "article_url",
    "source",
    "category",
    "summary",
]

news_classified_df.to_csv(classified_export_path, index=False)
news_prototype_df = news_classified_df.loc[
    news_classified_df["news_relevance"].eq("Included"), prototype_columns
].copy()
news_prototype_df.to_csv(prototype_export_path, index=False)

print(f"Classified export: {classified_export_path.relative_to(repo_root)} ({len(news_classified_df)} records)")
print(f"Prototype export: {prototype_export_path.relative_to(repo_root)} ({len(news_prototype_df)} records)")
print("\nPrototype fields:")
print(news_prototype_df.columns.tolist())
display(news_prototype_df.head())

### 6.1 Reproducibility Validation

In [ ]:
classified_export_df = pd.read_csv(classified_export_path)
prototype_export_df = pd.read_csv(prototype_export_path)

assert raw_sha256 == "5a4cf94543aca33dcbb38e74e8837672710fbc6626cad30d327aa7cc3dc6218b"
assert len(raw_news_df) == 25
assert len(news_cleaned_df) == 25
assert len(classified_export_df) == 25
assert len(news_prototype_df) == 22
assert len(prototype_export_df) == 22
assert set(prototype_export_df.columns) == set(prototype_columns)
assert set(news_classified_df["news_relevance"].unique()) == {"Included", "Excluded"}

print("Validation passed:")
print("- Raw RSS checksum matches the acquisition checksum.")
print("- Raw, cleaned and classified record counts are consistent.")
print("- Prototype export contains only Included records and source-supported metadata fields.")
print("- Repository-relative input and output paths are used.")